In [2]:
from datasets import load_from_disk
import datasets 
from transformers import AutoTokenizer 
import sys
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")
from data.CyberMetric import CyberMetric_evaluator
from openai import OpenAI 
import re
import pandas as pd 

df = pd.read_excel('data/class1_2_3_4_5.ods', engine='odf')

In [90]:
import numpy as np
class read_topic(): 
    def __init__(self, df): 
        ''' 
        Input:  dataframe of .ods CSEC2017 file, with ID, Knowledge_units, Topics, Description as headers
                & index  
        Output: topic, description, and CSEC2017 label of provided index or random index if index was not provided
        '''
        self.label = df.get(df.keys()[0])
        self.KU = df.get(df.keys()[1])
        self.topics = df.get(df.keys()[2])
        self.descriptions = df.get(df.keys()[3])

    def clean_text(self, text): 
        #special_chars = r'[\_\-]'
        #cleaned_text = re.sub(special_chars,' ', text)        # replace - or _ with space 
        #cleaned_text = re.sub(r'[^a-zA-Z0-9 ]', '', text)     # remove any non-alphanumeric /space character
        return text.lower() 
        
    def index(self, idx=None): 
        if idx == None:                                        # choose random index if index not provided
            idx = np.random.randint(0,len(self.label))
         
        description = self.clean_text(self.descriptions[idx])  # clean topics & descriptions 
        topic = self.clean_text(self.topics[idx])              # clean topics & descriptions 
        if (topic == ""):                                      # return knowledge unit if there is no topic
            topic = self.clean_text(self.KU[idx])
        
        if isinstance(self.label[idx],int):                    # label is either an int or a messy string
            label = str(self.label[idx])
        else: 
            label = self.clean_text(self.label[idx])
            
        return topic, description, label
        
df = pd.read_excel('data/class1_2_3_4_5.ods', engine='odf')
read = read_topic(df)
topic, description, label = read.index(0)
print(description)
# empty lines possible 

this topic covers basic concepts in cryptography to  build the base for other sections in the knowledge  unit. this topic includes: ● encryption/decryption, sender authentication,  data integrity, non-repudiation, ● attack classification (ciphertext-only, known  plaintext, chosen plaintext, chosen ciphertext),  ● secret key (symmetric), cryptography and publickey (asymmetric) cryptography,  ● information-theoretic security (one-time pad,  shannon theorem), and • computational security 


In [78]:
def submit_message_LLM(model, messages): 
    text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [80]:
# Load Qwen2-7B-Instruct model. 
from transformers import AutoModelForCausalLM, AutoTokenizer
API_KEY="<YOUR-APKI-KEY-HERE>"
model_path="Qwen/Qwen2.5-7B-Instruct"
device = "cuda" # the device to load the model onto

model2 = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [125]:
import json
from tqdm import tqdm
def topic_prompt(topic, description):
        """
        Formats a topic and description into a list of message dictionaries for the pipeline.
        """
        #options_str = ', '.join([f"{key}) {value}" for key, value in answers.items()])
        instructions = (
            "You are a helpful AI assistant.\n"
            "Instructions:\n"
            "a. Carefully read the topic and description .\n"
            "b. Give a list of all subtopics in the description .\n"
            "c. Do NOT include any explanation or additional text in the response.\n"
            #"d. Always return the answer in this XML format: '<xml>answer</xml>'. "
            #"For example, if the correct answer is D, then return <xml>D</xml>.\n\n"
        )
    
        messages = [
            {"role": "system", "content": instructions},
            {"role": "user", "content": f"#topic : {topic}\n #description: {description}"}
        ]
        return messages
    
# load topic, description and label of certain index using read_topic class 
idx = 0
read = read_topic(df)
topic, description, label = read.index(0)
print('description: ',description)
# create prompt based on topic + description
message = topic_prompt(topic, description)
#print('message: ',message)
# put the formatted prompt through the model: 
response = submit_message_LLM(model2, message)
print('response: ',response)

description:  this topic covers basic concepts in cryptography to  build the base for other sections in the knowledge  unit. this topic includes: ● encryption/decryption, sender authentication,  data integrity, non-repudiation, ● attack classification (ciphertext-only, known  plaintext, chosen plaintext, chosen ciphertext),  ● secret key (symmetric), cryptography and publickey (asymmetric) cryptography,  ● information-theoretic security (one-time pad,  shannon theorem), and • computational security 
response:  - Encryption/Decryption
- Sender Authentication
- Data Integrity
- Non-Repudiation
- Ciphertext-Only Attack
- Known Plaintext Attack
- Chosen Plaintext Attack
- Chosen Ciphertext Attack
- Secret Key (Symmetric) Cryptography
- Public Key (Asymmetric) Cryptography
- Information-Theoretic Security (One-Time Pad)
- Shannon Theorem
- Computational Security


In [135]:
# re-format output 
def clean_text(text): 
    special_chars = r'[\_]'
    cleaned_text = re.sub(special_chars,' ', text)        # replace - or _ with space 
    cleaned_text = re.sub(r'[^a-zA-Z0-9 -/()]', '', cleaned_text)     # remove any non-alphanumeric /space character
    special_chars = r'[\-]'
    cleaned_text = re.sub(special_chars,'', cleaned_text)        # replace - or _ with space 
    return cleaned_text.lower() 

subtopics = response.split("\n")
for sub in subtopics: 
    if sub[:2] == '- ':
        sub = sub[2:]
        print(sub)
    print(clean_text(sub))
print(label)

Encryption/Decryption
encryption/decryption
Sender Authentication
sender authentication
Data Integrity
data integrity
Non-Repudiation
nonrepudiation
Attack Classification (Ciphertext-Only, Known Plaintext, Chosen Plaintext, Chosen Ciphertext)
attack classification (ciphertextonly, known plaintext, chosen plaintext, chosen ciphertext)
Secret Key (Symmetric) Cryptography
secret key (symmetric) cryptography
Public-Key (Asymmetric) Cryptography
publickey (asymmetric) cryptography
Information-Theoretic Security (One-Time Pad, Shannon Theorem)
informationtheoretic security (onetime pad, shannon theorem)
Computational Security
computational security
1


In [136]:
# make new dataframe and save it
read = read_topic(df)
for idx in [0]: #range(len(read.labels)):
    # check if not empty! 
    topic, description, label = read.index(idx)
    print('description: ',description)
    # create prompt based on topic + description
    message = topic_prompt(topic, description)
    #print('message: ',message)
    # put the formatted prompt through the model: 
    response = submit_message_LLM(model2, message)
    print('response: ',response)
    subtopics = response.split("\n")
    for sub in subtopics: 
        if sub[:2] == '- ':
            sub = sub[2:]
            print(sub)
        print(clean_text(sub))
    print(label)

description:  this topic covers basic concepts in cryptography to  build the base for other sections in the knowledge  unit. this topic includes: ● encryption/decryption, sender authentication,  data integrity, non-repudiation, ● attack classification (ciphertext-only, known  plaintext, chosen plaintext, chosen ciphertext),  ● secret key (symmetric), cryptography and publickey (asymmetric) cryptography,  ● information-theoretic security (one-time pad,  shannon theorem), and • computational security 
response:  - Encryption/Decryption
- Sender Authentication
- Data Integrity
- Non-repudiation
- Ciphertext-only Attack
- Known Plaintext Attack
- Chosen Plaintext Attack
- Chosen Ciphertext Attack
- Secret Key (Symmetric) Cryptography
- Public Key (Asymmetric) Cryptography
- Information-Theoretic Security (One-Time Pad)
- Shannon Theorem
- Computational Security
Encryption/Decryption
encryption/decryption
Sender Authentication
sender authentication
Data Integrity
data integrity
Non-repudiat

### Old code

In [77]:
def clean_llm_output(text): 
    special_chars = r'[\"\[\]\']'
    cleaned_text = re.sub(special_chars, '', text)
    # replace - or _ with space 
    special_chars = r'[\_\-]'
    cleaned_text2 = re.sub(special_chars,' ', cleaned_text)
    return cleaned_text2 
strj = "This is \xa0 an-]'example"
output = clean_llm_output(strj)
print(output)

This is   an example
